In [3]:
import pandas as pd
from pathlib import Path

In [2]:
original_dataset = Path("original_dataset.csv")

In [3]:
df = pd.read_csv(original_dataset)

In [4]:
df.shape

(28587, 10)

In [5]:
df.isnull().sum()

subject     3838
body           0
answer         7
type           0
queue          0
priority       0
language       0
version        0
tag_1          0
tag_2         13
dtype: int64

In [6]:
df["type"].unique().tolist()

['Incident', 'Request', 'Problem', 'Change']

In [7]:
df["queue"].unique().tolist()

['Technical Support',
 'Returns and Exchanges',
 'Billing and Payments',
 'Sales and Pre-Sales',
 'Service Outages and Maintenance',
 'Product Support',
 'IT Support',
 'Customer Service',
 'Human Resources',
 'General Inquiry']

In [8]:
df["priority"].unique().tolist()

['high', 'medium', 'low']

In [9]:
df["language"].unique().tolist()

['de', 'en']

In [10]:
df["version"].unique().tolist()

[51, 52, 400]

In [11]:
df["tag_1"].unique().tolist()
df["tag_2"].unique().tolist()

['Outage',
 'Disruption',
 'Feature',
 'Payment',
 'Product',
 'Hardware',
 'Feedback',
 'Performance',
 'Crash',
 'Service',
 'Issue',
 'Account',
 'Maintenance',
 'Documentation',
 'Security',
 'Tech Support',
 'Compliance',
 'Bug',
 'Connectivity',
 'IT',
 'Investment',
 'Return',
 'Software',
 'Support',
 'Network',
 'Invoice',
 'Incident',
 'Refund',
 'Integration',
 'Discrepancy',
 'Lead',
 'Microservice',
 'Malware',
 'Analytics',
 'Login',
 'Coordination',
 'Dashboard',
 'Compatibility',
 'Virus',
 'Alert',
 'Optimization',
 'Firmware',
 'Workflow',
 'Swap',
 'System',
 'Marketing',
 'Email',
 'Usability',
 'Information',
 'Synchronization',
 'Sales',
 'Cloud Native',
 'Windows',
 'Cloud-Native',
 'VPN',
 'RAID',
 'Monitoring',
 'Financial Planning',
 'Shipment',
 'Data',
 'Employee',
 'Replacement',
 'Issue Resolution',
 'UI/UX',
 'Cloud',
 'Vulnerability',
 'Campaign',
 'Collaboration',
 'Exchange',
 'Policy',
 'Process',
 'Privacy',
 'Notification',
 'Admin',
 'Process Impro

In [12]:
df["queue"].value_counts(dropna=False)

queue
Technical Support                  8362
Product Support                    5252
Customer Service                   4268
IT Support                         3433
Billing and Payments               2788
Returns and Exchanges              1437
Service Outages and Maintenance    1148
Sales and Pre-Sales                 918
Human Resources                     576
General Inquiry                     405
Name: count, dtype: int64

In [13]:
target_queues = [
    'Technical Support',
    'Service Outages and Maintenance',
    'Product Support',
    'IT Support',
    'Customer Service'
]

for q in target_queues:
    print(f"\n{'='*50}\nQueue: {q}\n{'='*50}")
    for _, row in df[df['queue'] == q].head(5).iterrows():
        print(f"- Subject: {row['subject']}")
        print(f"  Type: {row['type']} | Priority: {row['priority']}")
        print(f"  Body: {row['body']}\n")


Queue: Technical Support
- Subject: Wesentlicher Sicherheitsvorfall
  Type: Incident | Priority: high
  Body: Sehr geehrtes Support-Team,\n\nich möchte einen gravierenden Sicherheitsvorfall melden, der gegenwärtig mehrere Komponenten unserer Infrastruktur betrifft. Betroffene Geräte umfassen Projektoren, Bildschirme und Speicherlösungen auf Cloud-Plattformen. Der Grund für die Annahme ist, dass der Vorfall eine potenzielle Datenverletzung im Zusammenhang mit einer Cyberattacke darstellt, was ein erhebliches Risiko für sensible Informationen und den laufenden Geschäftsbetrieb unserer Organisation bedeutet.\n\nUnsere initialen Untersuchungen haben ungewöhnliche Aktivitäten und Abweichungen bei den Geräten ergeben. Trotz der Umsetzung unserer standardisierten Behebungs- und Eindämmungsmaßnahmen konnte die Bedrohung bislang nicht vollständig eliminiert.

- Subject: Account Disruption
  Type: Incident | Priority: high
  Body: Dear Customer Support Team,\n\nI am writing to report a signific

In [14]:
target_queues = [
    "Technical Support",
    "IT Support",
    "Service Outages and Maintenance"
]

df= df[
    (df["language"] == "en") & 
    (df["queue"].isin(target_queues))
].copy()

In [15]:
df.shape

(7343, 10)

In [16]:
df.columns.tolist()

['subject',
 'body',
 'answer',
 'type',
 'queue',
 'priority',
 'language',
 'version',
 'tag_1',
 'tag_2']

In [17]:
columns_to_drop = [
    "answer",
    "queue",
    "priority",
    "language",
    "version",
    "tag_1",
    "tag_2"
]

df = df.drop(columns=columns_to_drop)

In [18]:
df.shape

(7343, 3)

In [19]:
df.columns.tolist()

['subject', 'body', 'type']

In [20]:
df["ticket"] = (df["subject"].fillna("") + "\n\n" + df["body"]).str.strip()

In [21]:
df = df.drop(columns=["subject", "body"])

In [22]:
df["complexity"] = None

In [24]:
df["ticket"].duplicated().sum()

np.int64(0)

In [26]:
rules_path = Path("annotation_rules.md")
system_prompt = rules_path.read_text(encoding="utf-8")

In [27]:
import json
import urllib.request
import urllib.error

In [28]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "deepseek-r1:8b"
OUTPUT_FILE = "annotated_dataset.csv"

In [30]:
def annotate_ticket(idx: int, ticket_type: str, ticket_text: str, rules: str) -> tuple:
    user_content = f"type: {ticket_type}\n\nticket:\n{ticket_text}"
    
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": rules},
            {"role": "user", "content": user_content}
        ],
        "format": "json",
        "stream": False,
        "options": {
            "temperature": 0.0
        }
    }
    
    req = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read().decode("utf-8"))
            content = json.loads(data["message"]["content"])
            
            label = content.get("label", "").strip()
            reasoning = content.get("reasoning", "").strip()
            return idx, label, reasoning, None
    except Exception as e:
        return idx, None, None, str(e)

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# Find unannotated rows (allows resuming if stopped)
pending_indices = df[df["complexity"].isna()].index.tolist()
print(f"Starting annotation on {len(pending_indices):,} pending rows with 2 parallel threads...")

lock = threading.Lock()
completed_count = 0
failed_count = 0

with ThreadPoolExecutor(max_workers=2) as executor:
    # Submit pending rows to 2 threads
    futures = {
        executor.submit(
            annotate_ticket,
            idx,
            str(df.at[idx, "type"]),
            str(df.at[idx, "ticket"]),
            system_prompt
        ): idx
        for idx in pending_indices
    }
    
    with tqdm(total=len(pending_indices), desc="Annotating tickets") as pbar:
        for future in as_completed(futures):
            idx, label, reasoning, error = future.result()
            
            with lock:
                if error is None and label:
                    df.at[idx, "complexity"] = label
                    df.at[idx, "reasoning"] = reasoning
                else:
                    failed_count += 1
                
                completed_count += 1
                pbar.update(1)
                
                # Export every 50 tickets
                if completed_count % 50 == 0:
                    df.to_csv(OUTPUT_FILE, index=False)
                    pbar.set_postfix({
                        "saved": completed_count,
                        "failed": failed_count
                    })

# Final save
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nDone! Exported full dataset to {OUTPUT_FILE}")
print(f"Total completed: {completed_count:,} | Errors: {failed_count:,}")

In [4]:
annotated_df = pd.read_csv(
    Path("annotated_dataset.csv")
)

labels = ["single_hop", "multi_hop", "parallel_hop"]
counts = annotated_df["complexity"].value_counts().reindex(labels, fill_value=0)

distribution = pd.DataFrame({
    "count": counts,
    "percentage": (counts / counts.sum() * 100).round(2)
})

display(distribution)

,count,percentage
complexity,,
single_hop,576,32.16
multi_hop,1150,64.21
parallel_hop,65,3.63
